# Capybara RC3 — Integrated Preservation + Expansion Deployment

RC3 upgrades the still-live Alpha documentation **in place** into the Original Edition 1.0.0. It recovers the exact public source/assets before any push, retains the Alpha visual language as the primary presentation, integrates expanded guidance into the same routes, adds genuinely new sections to the same navigation, audits completeness, pushes `main`, rebuilds the existing Read the Docs project, and verifies protected visual behaviors after publication.


In [1]:
from pathlib import Path
import subprocess, sys, json, time, getpass, requests
REPO_ROOT=Path.cwd().resolve()
if not (REPO_ROOT/'.readthedocs.yaml').exists():
    raise RuntimeError('Open/run this notebook from the RC3 repository root.')
GITHUB_OWNER='BrianBowers-NapaCounty'
GITHUB_REPO='itam-itsm_integration_framework'
GITHUB_URL=f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
RTD_PROJECT_SLUG='capybara-framework'
RTD_VERSION='latest'
print('RC3 repository:',REPO_ROOT)
print('GitHub:',GITHUB_URL)
print('RTD:',RTD_PROJECT_SLUG)


RC3 repository: C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC3_Integrated
GitHub: https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git
RTD: capybara-framework


## 1. Recover Alpha and integrate RC3

Run this **before** the old public build is replaced. If an open Selenium `driver` exists in this kernel, the finalizer uses it as a fallback for public-file retrieval.


In [2]:
sys.path.insert(0,str(REPO_ROOT/'tools'))
from finalize_rc3 import finalize
report=finalize(REPO_ROOT,driver=globals().get('driver'))
if report['errors']:
    print(json.dumps(report['errors'][:30],indent=2))
    raise RuntimeError(f"RC3 finalization has {len(report['errors'])} recovery error(s). Nothing will be pushed.")
print('✓ Exact live Alpha baseline recovered and integrated.')


SOURCE 01/59: branding.rst.txt
SOURCE 02/59: changelog.md.txt
SOURCE 03/59: code-of-conduct.md.txt
SOURCE 04/59: contributing.md.txt
SOURCE 05/59: editions/alpha/0-overview.md.txt
SOURCE 06/59: editions/alpha/0-overview/0-1-introduction.md.txt
SOURCE 07/59: editions/alpha/0-overview/0-2-problem-statement.md.txt
SOURCE 08/59: editions/alpha/0-overview/0-3-executive-summary.md.txt
SOURCE 09/59: editions/alpha/0-overview/README.md.txt
SOURCE 10/59: editions/alpha/1-implementation-guide.md.txt
SOURCE 11/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-1-prerequisites.md.txt
SOURCE 12/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-2-budget-procurement.md.txt
SOURCE 13/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-3-deployment-steps.md.txt
SOURCE 14/59: editions/alpha/1-implementation-guide/1-1-first-steps/1-1-4-configuration.md.txt
SOURCE 15/59: editions/alpha/1-implementation-guide/1-1-first-steps/README.md.txt
SOURCE 16/59: editions/alpha/1-impl

In [3]:
audit=subprocess.run([
    sys.executable,str(REPO_ROOT/'tools/audit_rc3.py'),
    '--repo',str(REPO_ROOT),'--strict'
],text=True)
if audit.returncode!=0:
    raise RuntimeError('RC3 completeness audit failed. Nothing will be pushed.')
print('✓ RC3 completeness audit PASS')
print('Evidence: legacy_baseline/RC3_COVERAGE_MATRIX.csv')


RuntimeError: RC3 completeness audit failed. Nothing will be pushed.

## 2. Commit and push the audited integrated source


In [ ]:
def git(*args,check=True):
    r=subprocess.run(['git','-C',str(REPO_ROOT),*args],capture_output=True,text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r

if git('rev-parse','--git-dir',check=False).returncode!=0:
    git('init')
remotes=git('remote',check=False).stdout.split()
if 'origin' in remotes:
    git('remote','set-url','origin',GITHUB_URL)
else:
    git('remote','add','origin',GITHUB_URL)

git('add','-A')
if git('rev-parse','--verify','HEAD',check=False).returncode!=0:
    git('commit','-m','Publish Capybara Original Edition 1.0.0 RC3')
elif git('status','--porcelain',check=False).stdout.strip():
    git('commit','-m','Integrate complete Alpha baseline and expanded RC3 documentation')

git('branch','-M','main')
confirm=input(f'Type PUSH {GITHUB_OWNER}/{GITHUB_REPO} to publish the audited RC3 source: ').strip()
if confirm!=f'PUSH {GITHUB_OWNER}/{GITHUB_REPO}':
    raise RuntimeError('Push cancelled.')
git('push','-u','origin','main')
verify=git('ls-remote','--heads','origin','refs/heads/main')
if not verify.stdout.strip():
    raise RuntimeError('GitHub does not advertise refs/heads/main after push.')
print('✓ GitHub main:',verify.stdout.split()[0])


## 3. Reconnect, synchronize, and build the existing Read the Docs project


In [ ]:
RTD_API_TOKEN=globals().get('RTD_API_TOKEN','').strip()
if not RTD_API_TOKEN:
    RTD_API_TOKEN=getpass.getpass('Read the Docs API token (hidden): ').strip()
    globals()['RTD_API_TOKEN']=RTD_API_TOKEN
if not RTD_API_TOKEN:
    raise RuntimeError('No RTD API token supplied.')
API='https://app.readthedocs.org/api/v3'
def rtd(method,path,body=None,allowed=(200,201,202,204)):
    url=API.rstrip('/')+'/'+path.lstrip('/')
    headers={'Accept':'application/json','Authorization':f'Token {RTD_API_TOKEN}'}
    if body is not None: headers['Content-Type']='application/json'
    r=requests.request(method.upper(),url,headers=headers,json=body,timeout=60)
    try: data=r.json() if r.text else None
    except Exception: data=r.text
    if r.status_code not in allowed:
        raise RuntimeError({'status':r.status_code,'url':url,'data':data})
    return r.status_code,data

_,project=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/')
print('Current RTD repository:',project.get('repository',{}).get('url'))
confirm=input(f'Type BUILD {RTD_PROJECT_SLUG} to reconnect/sync/build latest: ').strip()
if confirm!=f'BUILD {RTD_PROJECT_SLUG}':
    raise RuntimeError('RTD build cancelled.')
rtd('PATCH',f'projects/{RTD_PROJECT_SLUG}/',{
    'repository':{'url':GITHUB_URL.removesuffix('.git'),'type':'git'},
    'default_branch':'main','default_version':'latest',
    'readthedocs_yaml_path':'.readthedocs.yaml'})
rtd('POST',f'projects/{RTD_PROJECT_SLUG}/sync-versions/')
time.sleep(5)
_,version=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/')
if not version.get('active') or version.get('hidden'):
    rtd('PATCH',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/',{'active':True,'hidden':False})
rtd('POST',f'projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/builds/')
print('✓ Build requested')


In [ ]:
def build_state(b):
    s=b.get('state')
    return (s.get('code') or s.get('name')) if isinstance(s,dict) else str(s or '')

build_id=None
for _ in range(40):
    _,data=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/builds/?limit=20')
    for b in data.get('results',[]):
        v=b.get('version'); v=v.get('slug') if isinstance(v,dict) else v
        if v==RTD_VERSION:
            build_id=b.get('id'); break
    if build_id: break
    time.sleep(2)
if not build_id:
    raise RuntimeError('Could not identify the latest RTD build.')
last=None
for _ in range(180):
    _,b=rtd('GET',f'projects/{RTD_PROJECT_SLUG}/builds/{build_id}/?expand=config')
    state=build_state(b).lower()
    if state!=last:
        print(time.strftime('%H:%M:%S'),state,'success=',b.get('success'))
        last=state
    if state in ('finished','cancelled'): break
    time.sleep(5)
else:
    raise RuntimeError('RTD build polling timed out.')
if not b.get('success'):
    if globals().get('driver'):
        driver.get(f'https://app.readthedocs.org/projects/{RTD_PROJECT_SLUG}/builds/{build_id}/')
    raise RuntimeError(f'RTD build {build_id} failed; build page opened if Selenium is available.')
print('✓ RTD build succeeded:',build_id,'commit=',b.get('commit'))


## 4. Verify the published look and protected media behavior


In [ ]:
docs_url=f'https://{RTD_PROJECT_SLUG}.readthedocs.io/en/latest/'
if globals().get('driver'):
    from verify_live_visual_behavior import verify
    verify(driver,REPO_ROOT)
    driver.get(docs_url)
else:
    print('No Selenium driver in this kernel; browser-rendered visual audit skipped.')
    print('Run verify_live_visual_behavior.verify(driver, REPO_ROOT) before declaring RC3 final.')
print('LIVE:',docs_url)
